# 🌿 Plant Disease Detection using Convolutional Neural Network (CNN)

* Developed a deep learning model to classify **38 plant disease categories** from leaf images.

* Implemented a **custom CNN architecture** with Conv2D, BatchNormalization, MaxPooling, and Dense layers.

* Trained on the **PlantVillage Dataset** — 87,000+ labeled leaf images across 14 crop species.

* Used **Data Augmentation** to improve generalization and reduce overfitting.

* Visualized results using **Training Curves**, **Confusion Matrix**, and **Sample Predictions**.

---
**Dataset:** [PlantVillage Disease Dataset – Kaggle](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset)

**Setup:** Download the dataset and set `DATASET_PATH` to the `plantvillage dataset/color` folder.

# Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dense, Dropout, GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)
print("All libraries imported successfully!")

# Configuration & Hyperparameters

In [ ]:
# --- Dataset Path ---
# Update this to your local path or Google Drive path
DATASET_PATH = "/content/plantvillage dataset/color"

# --- Image Parameters ---
IMG_HEIGHT   = 128
IMG_WIDTH    = 128
IMG_SIZE     = (IMG_HEIGHT, IMG_WIDTH)
CHANNELS     = 3

# --- Training Parameters ---
BATCH_SIZE   = 32
EPOCHS       = 30
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2

# Seed for reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("Configuration set!")
print(f"Image size    : {IMG_SIZE}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Max epochs    : {EPOCHS}")

# Dataset Exploration

In [ ]:
# List all class folders
class_names = sorted(os.listdir(DATASET_PATH))
NUM_CLASSES  = len(class_names)

print(f"Total classes  : {NUM_CLASSES}")
print(f"\nSample classes :")
for i, c in enumerate(class_names[:10]):
    count = len(os.listdir(os.path.join(DATASET_PATH, c)))
    print(f"  [{i:02d}] {c:<50} → {count} images")
print("  ...")

In [ ]:
# Count images per class
class_counts = {cls: len(os.listdir(os.path.join(DATASET_PATH, cls))) for cls in class_names}
total_images = sum(class_counts.values())

print(f"Total images in dataset: {total_images:,}")

# Bar plot of image counts
plt.figure(figsize=(18, 5))
plt.bar(range(NUM_CLASSES), class_counts.values(), color='steelblue', edgecolor='black')
plt.xticks(range(NUM_CLASSES), class_names, rotation=90, fontsize=7)
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.title('Image Count per Class (PlantVillage Dataset)')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample images from 9 random classes
import random
random.seed(SEED)
sample_classes = random.sample(class_names, 9)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, cls in zip(axes.flatten(), sample_classes):
    cls_path = os.path.join(DATASET_PATH, cls)
    img_name = random.choice(os.listdir(cls_path))
    img      = mpimg.imread(os.path.join(cls_path, img_name))
    ax.imshow(img)
    ax.set_title(cls.replace('___', '\n').replace('_', ' '), fontsize=7)
    ax.axis('off')

plt.suptitle('Sample Leaf Images from PlantVillage Dataset', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Data Augmentation & Generators

In [ ]:
# Training generator — with augmentation
train_datagen = ImageDataGenerator(
    rescale           = 1.0 / 255,
    validation_split  = VALIDATION_SPLIT,
    rotation_range    = 30,
    width_shift_range = 0.15,
    height_shift_range= 0.15,
    shear_range       = 0.15,
    zoom_range        = 0.15,
    horizontal_flip   = True,
    fill_mode         = 'nearest'
)

# Validation generator — only rescale, no augmentation
val_datagen = ImageDataGenerator(
    rescale          = 1.0 / 255,
    validation_split = VALIDATION_SPLIT
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    subset       = 'training',
    seed         = SEED,
    shuffle      = True
)

val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    subset       = 'validation',
    seed         = SEED,
    shuffle      = False
)

# Save class mapping
class_indices = train_generator.class_indices
idx_to_class  = {v: k for k, v in class_indices.items()}

print(f"\nTraining batches  : {len(train_generator)}")
print(f"Validation batches: {len(val_generator)}")

In [ ]:
# Visualize augmented images from training generator
x_batch, y_batch = next(train_generator)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, lbl in zip(axes.flatten(), x_batch[:8], y_batch[:8]):
    ax.imshow(img)
    ax.set_title(idx_to_class[np.argmax(lbl)].split('___')[-1].replace('_', ' '), fontsize=8)
    ax.axis('off')

plt.suptitle('Augmented Training Images', fontsize=13)
plt.tight_layout()
plt.show()

# Build CNN Model

In [ ]:
def build_cnn(num_classes, img_height=128, img_width=128):
    model = Sequential([

        # ── Block 1 ──
        Conv2D(32, (3, 3), activation='relu', padding='same',
               input_shape=(img_height, img_width, 3)),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # ── Block 2 ──
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # ── Block 3 ──
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # ── Block 4 ──
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # ── Classifier Head ──
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    return model


model = build_cnn(NUM_CLASSES)
model.compile(
    optimizer = Adam(learning_rate=LEARNING_RATE),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
model.summary()

# Callbacks

In [ ]:
callbacks = [
    EarlyStopping(
        monitor           = 'val_loss',
        patience          = 7,
        restore_best_weights = True,
        verbose           = 1
    ),
    ModelCheckpoint(
        'best_plant_cnn.keras',
        monitor           = 'val_accuracy',
        save_best_only    = True,
        verbose           = 1
    ),
    ReduceLROnPlateau(
        monitor           = 'val_loss',
        factor            = 0.5,
        patience          = 3,
        min_lr            = 1e-6,
        verbose           = 1
    )
]

print("Callbacks configured!")
print("  - EarlyStopping   (patience=7)")
print("  - ModelCheckpoint (saves best val_accuracy model)")
print("  - ReduceLROnPlateau (halves LR when val_loss stalls)")

# Train Model

In [ ]:
history = model.fit(
    train_generator,
    epochs              = EPOCHS,
    validation_data     = val_generator,
    callbacks           = callbacks,
    verbose             = 1
)

# Plot Training Curves

In [ ]:
# Accuracy
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train Accuracy',      color='steelblue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orangered')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Loss
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train Loss',      color='steelblue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orangered')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Learning Rate (if ReduceLROnPlateau was used)
if 'lr' in history.history:
    plt.figure(figsize=(8, 3))
    plt.plot(history.history['lr'], color='green', label='Learning Rate')
    plt.xlabel('Epochs')
    plt.ylabel('LR')
    plt.title('Learning Rate Schedule')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Evaluate Model

In [ ]:
val_generator.reset()
val_loss, val_acc = model.evaluate(val_generator, verbose=0)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc:.4f} ({val_acc*100:.2f}%)")

# Predictions & Classification Report

In [ ]:
val_generator.reset()
y_pred_proba = model.predict(val_generator, verbose=1)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = val_generator.classes

print(f"\nTotal predictions: {len(y_pred)}")

In [ ]:
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

report = classification_report(y_true, y_pred, target_names=target_names)
print(report)

# Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(20, 18))
sns.heatmap(
    cm,
    annot     = True,
    fmt       = 'd',
    cmap      = 'Greens',
    xticklabels = [c.split('___')[-1] for c in target_names],
    yticklabels = [c.split('___')[-1] for c in target_names],
    linewidths  = 0.5
)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix – Plant Disease Detection CNN', fontsize=14)
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.show()

# Visualize Sample Predictions

In [ ]:
val_generator.reset()
x_batch, y_batch = next(val_generator)
preds = model.predict(x_batch, verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, img, true_lbl, pred_proba in zip(axes.flatten(), x_batch, y_batch, preds):
    true_idx  = np.argmax(true_lbl)
    pred_idx  = np.argmax(pred_proba)
    true_name = idx_to_class[true_idx].split('___')[-1].replace('_', ' ')
    pred_name = idx_to_class[pred_idx].split('___')[-1].replace('_', ' ')
    conf      = pred_proba[pred_idx] * 100
    color     = 'green' if true_idx == pred_idx else 'red'

    ax.imshow(img)
    ax.set_title(
        f"True : {true_name}\nPred : {pred_name} ({conf:.1f}%)",
        fontsize=7, color=color
    )
    ax.axis('off')

plt.suptitle('Sample Predictions  (Green = Correct | Red = Wrong)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Predict on a Single Custom Image

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

def predict_disease(img_path, model, idx_to_class, img_size=(128, 128)):
    """
    Predict plant disease from a single image file.
    """
    img    = keras_image.load_img(img_path, target_size=img_size)
    arr    = keras_image.img_to_array(img) / 255.0
    arr    = np.expand_dims(arr, axis=0)

    proba  = model.predict(arr, verbose=0)[0]
    idx    = np.argmax(proba)
    label  = idx_to_class[idx]
    conf   = proba[idx] * 100

    # Show image
    plt.figure(figsize=(4, 4))
    plt.imshow(keras_image.load_img(img_path, target_size=img_size))
    plt.axis('off')
    plt.title(f"{label.replace('___', ' – ').replace('_', ' ')}\nConfidence: {conf:.1f}%",
              fontsize=10, color='green')
    plt.tight_layout()
    plt.show()

    return label, conf


# ---- Example usage ----
# Replace with any leaf image path
# label, conf = predict_disease("/content/sample_leaf.jpg", model, idx_to_class)
# print(f"Predicted : {label}")
# print(f"Confidence: {conf:.2f}%")

print("Function `predict_disease` is ready.")
print("Uncomment and replace the path above with your own leaf image to test!")

# Top-5 Predictions for a Sample

In [ ]:
def top_k_predictions(img_array, model, idx_to_class, k=5):
    """
    Show top-k disease predictions with confidence.
    img_array: preprocessed image, shape (H, W, 3), values 0–1
    """
    inp   = np.expand_dims(img_array, axis=0)
    proba = model.predict(inp, verbose=0)[0]
    top_k = np.argsort(proba)[::-1][:k]

    print(f"{'Rank':<6} {'Class':<50} {'Confidence':>10}")
    print("-" * 70)
    for rank, idx in enumerate(top_k, 1):
        name = idx_to_class[idx].replace('___', ' – ').replace('_', ' ')
        print(f"  #{rank}   {name:<48} {proba[idx]*100:>8.2f}%")


# Test on first validation image
val_generator.reset()
x_batch, _ = next(val_generator)
print("Top-5 predictions for validation image #0:\n")
top_k_predictions(x_batch[0], model, idx_to_class, k=5)

# Save Final Model

In [ ]:
model.save('plant_disease_cnn_final.keras')
print("Model saved as 'plant_disease_cnn_final.keras'")

# Optional: save class mapping
import json
with open('class_mapping.json', 'w') as f:
    json.dump(idx_to_class, f, indent=2)
print("Class mapping saved as 'class_mapping.json'")